# Solving the finite LP

Assumed as given:
- a finite set of the joint action space (from the separation oracle)
- amount of generators (from the market)
- a value for omega (from the golden section search)
- a socially optimal joint action (should be computed!)
- a market clearing function that maps joint actions to prices, costs, and profits (from the market)
- a market with some network and capacity constraints? (Needed in the market clearing function to properly clear the market)
- a function to replace an action of a joint action with a new action

Needs to be defined by the team:
- data structure for the actions, and how to replace an action in a joint action
- data structure for the market which includes number of generators and market clearing function giving prices, costs, and profits

Can be optimized by:
- Only need to add a new constraint to the LP for each new joint action that is generated by the separation oracle. This means that the LP can be solved iteratively, adding constraints as needed based on the actions generated.

In [ ]:
from pyomo.environ import *

# joint action set (set of tuple of bids)
A = Set()

# amount of generators
generators = 2

# value for omega
w = 1

# social optimal joint action
a_true = None

# social cost function
def C(a):
    return 1

# individual cost function (negative profit)
def c(a, i):
    return 0

# Replace action i in the joint action a with new action a_i
def replace_action(a, i, new_a_i):
    return None


# LP model:
model = ConcreteModel()

# parameters
model.generators = Set(initialize=range(generators))
model.A = Set(initialize=A)
model.a_true = Param(initialize=a_true)

# variables
model.p = Var(domain=NonNegativeReals)
model.v = Var(domain=NonNegativeReals)

# objective
model.objective = Objective(expr=model.p, sense=minimize)

# primary constraint
def constraint_rule(model, a):
    return C(a) + model.v * sum(c(replace_action(a,i,model.w * model.a_true[i]),i) - c(a, i) for i in model.generators) <= model.p * C(model.a_true)

model.constraint = Constraint(model.A, rule = constraint_rule)



In [5]:
def runModel(model, addprint = False):
    solver = SolverFactory("gurobi")
    if not solver.available():
        raise RuntimeError("Gurobi solver is not available in your environment.")
    results = solver.solve(model, tee=addprint)
    status = results.Solver()['Termination condition'].value
    assert status == 'optimal', f'error occurred, status: {status}.  Check model!'
    return model